<center>

# Universidad Nacional de Lomas de Zamora
## Facultad de Ingeniería
### Proyecto FINAL - Estacion de calidad por Perfilómetria
#### Alumno: QUINTANA, Fernando Miguel

</center>

In [ ]:
import numpy as np

# --------------------------------------
# GeometrySpace
# --------------------------------------

# Planos y rectas en el espacio euclídeo 3D
# https://www2.eii.uva.es/fund_inf/python/notebooks/Bibliotecas/03_Numpy/Numpy.html

def normaliza(v, tol=1e-6):
    """
    Devuelve el vector normalizado de v.

    Parameters
    ----------
    v : list, tuple, numpy.ndarray: numérico
        Vector unidimensional
    tol : float
        Tolerancia que debe cumplir la norma del vector v
    Returns
    -------
    v_norm : numpy.ndarray of floats
        Vector normalizado unitario
    Raises
    ------
    ValueError
        Si la norma del vector inicial es inferior a tol
    Example
    -------
        >>> v = (1, 1, 1)
        >>> normaliza(v)
    """

    norma = np.linalg.norm(v)
    if norma < tol:
        raise ValueError("La norma del vector es inferior a la tolerancia {}.".format(tol))
    v_norm = v / norma  # Al usar np.linalg.norm() v_norm es un numpy.ndarray
    return v_norm


def distancia_entre_puntos(p1, p2):
    """
    Calcula la distancia entre dos puntos en el espacio.

    Parameters
    ----------
    p1, p2 : numpy.ndarray
        Vectores unidimensionales
    Returns
    -------
    dist : numpy.float
        Distancia entre los puntos p1 y p2
    Example:
    --------
    >>> p1 = np.array((1, 1, 1), dtype=float)
    >>> p2 = np.array((2, 3, 4), dtype=float)
    >>> distancia_entre_puntos(p1, p2)
    """
    dist = np.linalg.norm(p1 - p2)  # p1-p2 no está definido para tuplas o listas
    return dist


def angulo(v1, v2):
    """
    Calcula el ángulo en radianes entre dos vectores.

    Parameters
    ----------
    v1, v2 : tuple, list, numpy.ndarray: numérico
        Vectores unidimensionales
    Returns
    -------
    angulo : float
        Ángulo en radianes que forman los vectores
    Example
    -------
    >>> u_x = [1, 0, 0]  # Vector unitario eje X
    >>> u_y = [0, 1, 0]  # Vector unitario eje Y
    >>> angulo(u_x, u_y)*180/np.pi  # Ángulo en grados
    """

    prod_esc = np.dot(v1, v2)
    norma_v1 = np.linalg.norm(v1)
    norma_v2 = np.linalg.norm(v2)

    angulo = np.arccos(np.clip(prod_esc / (norma_v1 * norma_v2), -1, 1))
    return angulo


def plano_tres_puntos(p1, p2, p3, tol=1e-3):
    """
    Calcula el plano definido por tres puntos en el espacio.

    Parameters
    ----------
    p1, p2, p3 : numpy.ndarray of floats
        Vectores unidimensionales representando 3 puntos 3D
    tol : float
        Tolerancia que deben cumplir las distancias entre los puntos
    Returns
    -------
    n, D : tuple : (numpy.ndarray of float, numpy float)
        n es el vector unitario perpendicular al plano
        D es el término independiente de la ecuación algebraica Ax+By+Cz+D=0
    Raises
    ------
    ValueError
        Si la distancia entre alguno de los 3 puntos es inferior a tol
    Example
    -------
    >>> p1 = np.array((1, 0, 0))
    >>> p2 = np.array((0, 1, 0))
    >>> p3 = np.array((0, 0, 1))
    >>> n, D = plano_tres_puntos(p1, p2, p3)
    """
    dist12 = distancia_entre_puntos(p1, p2)
    dist13 = distancia_entre_puntos(p1, p3)
    dist23 = distancia_entre_puntos(p2, p3)

    if dist12 < tol or dist13 < tol or dist23 < tol:
        raise ValueError("Distancia entre puntos menor que la tolerancia {}.".format(tol))

    # Producto vectorial para A B C
    # |  i       j       k    |
    # |v_12[0] v_12[1] v_12[2]|
    # |v_13[0] v_13[1] v_13[2]|

    v12 = p1 - p2
    v13 = p1 - p3

    n = normaliza(np.cross(v12, v13))

    D = -np.dot(n, p1)

    return n, D


def recta_dos_puntos(p1, p2, tol=1e-3):
    """
    Parameters
    ----------
    p1, p2 : numpy.ndarray of floats
        Vectores unidimensionales representando 2 puntos 3D
    tol : float
        Tolerancia que debe cumplir la distancia entre los puntos
    Returns
    -------
    p, v : tuple: (numpy.ndarray of float, numpy.ndarray of float)
        p es una copia de p1, punto pasado como primer parámetro

        v es el vector director unitario de la recta
    Raises
    ------
    ValueError
        Si la distancia entre los 2 puntos es inferior a tol
    Example
    -------
    >>> p1 = np.array((2, 2, 2))
    >>> p2 = np.array((3, 3, 3))
    >>> p, v = recta_dos_puntos(p1, p2)
    """
    if distancia_entre_puntos(p1, p2) < tol:
        raise ValueError("Distancia entre puntos menor que la tolerancia {}.".format(tol))
    v = normaliza(p1 - p2)
    return np.copy(p1), v


def interseccion_recta_plano(recta, plano, tol=0.0005):
    """
    Parameters
    ----------
    recta : tuple : (numpy.ndarray of floats, numpy.ndarray of floats)
        Recta definida por la tupla punto inicial y vector director
        p=p_0+l*v
    plano : tuple : (numpy.ndarray of floats, float)
        Plano definido por la tupla vector normal (A, B, C) y término independiente D
        Ax+By+Cz+D=0
    tol : float
        Tolerancia que deben cumplir el ángulo entre recta y plano
    Returns
    -------
    p : numpy.ndarray of floats
        El punto de intersección
    Raises
    ------
    ValueError
        Si el ángulo entre ambas figuras es inferior a tol en radianes
    Example
    -------
    >>> p1 = np.array((1, 0, 0))
    >>> p2 = np.array((0, 1, 0))
    >>> p3 = np.array((0, 0, 1))
    >>> p4 = np.array((2, 2, 2))
    >>> p5 = np.array((3, 3, 3))
    >>> plano = plano_tres_puntos(p1, p2, p3)
    >>> recta = recta_dos_puntos(p4, p5)
    >>> p = interseccion_recta_plano(recta, plano)
    """
    ang = angulo(recta[1], plano[0])

    if np.abs(ang - np.pi / 2) < tol:
        print(np.abs(ang - np.pi / 2))
        raise ValueError("La recta y el plano son casi paralelos según tolerancia {}.".format(tol))

    lamb = -(np.dot(recta[0], plano[0]) + plano[1]) / np.dot(recta[1], plano[0])

    return recta[0] + lamb * recta[1]


# --------------------------------------
# GeneradorMatriz
# --------------------------------------

# punto focal
p_focal = np.array((-96.10, 166.45, 0))

# punto plano del laser
p_laser_1 = np.array((0, 0, 0))
p_laser_2 = np.array((0, 1, 0))
p_laser_3 = np.array((0, 0, 1))

# Datos cámara
px_wh = 0.003     # mm tamaño de alto - ancho de cada pixel
focal_dist = 7.8  # mm distancia focal de cámara
W = 732           # Referencia pixeles verticales -> range(0, 733):
H = 1296          # Referencia pixeles horizontales -> range(-648, 649):

# Creamos el plano del láser
plano_laser = plano_tres_puntos(p1=p_laser_1, p2=p_laser_2, p3=p_laser_3)

# Recorremos cada pixel del sensor de la cámara
mtx_ppx = []
mtx_ppy = []
mtx_ppz = []

for fpx in range(0, 733):
    aux_x = []
    aux_y = []
    aux_z = []

    # Posición de cada pixel en X e Y (varía con la fila)
    pos_x = fpx * px_wh * np.cos(30 * np.pi / 180) - (100 + 360 * px_wh * np.cos(30 * np.pi / 180))
    pos_y = fpx * px_wh * np.sin(30 * np.pi / 180) + (173.21 - 360 * px_wh * np.sin(30 * np.pi / 180))

    # Recorremos por columna del sensor
    for cpx in range(-648, 649):
        if cpx == 0:
            pass
        else:
            pos_z = cpx * px_wh

            # Creamos la recta del laser
            px_pos = np.array((pos_x, pos_y, pos_z))
            recta_haz = recta_dos_puntos(px_pos, p_focal)

            # Obtenemos intersección recta-plano
            intersection = interseccion_recta_plano(recta=recta_haz, plano=plano_laser)

            # Guardamos la posición X,Y,Z en el espacio para ese pixel
            XX = np.around(intersection, 3).tolist()[0]
            YY = np.around(intersection, 3).tolist()[1]
            ZZ = np.around(intersection, 3).tolist()[2]

            #print(px_pos)
            print("Para el pixel en el punto " + str(px_pos) + " --- > " + str(XX) + ";" + str(YY) + ";" + str(ZZ))
            aux_x.append(XX)
            aux_y.append(YY)
            aux_z.append(ZZ)
            
    mtx_ppx.append(aux_x)
    mtx_ppy.append(aux_y)
    mtx_ppz.append(aux_z)

np.savetxt("mtx_x_izq.txt", mtx_ppx)
np.savetxt("mtx_y_izq.txt", mtx_ppy)
np.savetxt("mtx_z_izq.txt", mtx_ppz)
